This notebook is to inspect the way Zerlaut fits transfer function

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json
import time
import os
import pickle
import multiprocessing as mp
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

In [ ]:
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))

if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/haman/mf-temporary/MeanFieldTester')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')

import codes.controller as rw
from codes.plotting import fig_plots
import codes.cell_library as cells
import codes.neuron_simulation as ns
import codes.meanfield_simulation as mfs
import codes.network_simulation as nets
import codes.transfer_function as tf
from codes import utils

from codes.tvb_models.models import Zerlaut_adaptation_first_order


In [ ]:
workflow_params = {
    "single_neuron_simulations" : {
        "simulate_single_neuron" : False,
        "load_single_neuron" : True,
        "fix_nu_out" : True,
        "nu_e_range" : [0, 60, 16],
        "nu_i_range" : [0, 60, 16],
        "nu_out_range" : [0, 60, 16],
        "max_nu_e_step" : 4,
        "simulation_time" : 5000,
        "averaging_window" : 2000,
        "time_step" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "cpus" : 16
    },
    "transfer_functions" : {
        "fit_transfer_function" : True,
        "square_terms" : True,
        "log_term" : True,
        "adaptation" : False,
        "fit_with_w" : True,
        "nu_out_min" : 0.0,
        "nu_out_max" : 200.0,
        "V_eff_fitting" : {
            "method" : "SLSQP",
            "options" : {
                "ftol" : 5e-9,
                "disp" : false,
                "maxiter" : 10000
            }
        },
        "TF_fitting" : {
            "method" : "nelder-mead",
            "options" : {
                "fatol": 5e-9,
                "disp" : false,
                "maxiter" : 10000
            }
        }
    },
    "network_simulations" : {
        "load" : true,
        "timestep" : 0.1,
        "seed" : 42,
        "n_runs" : 5,
        "smoothing_function" : "sliding_window",
        "smoothing_constant" : 50,
        "smoothing_kwargs" : {}
    },
    "mf_model" : {
        "T" : 40.0,
        "tvb_model" : "DiVolo_Tsodyks_second_order",
        "mf_init" : {
            "E" : [0.000, 0.000],
            "I" : [0.00, 0.00],
            "C_ee" : [0.0000005, 0.0000005],
            "C_ei" : [0.0000005, 0.0000005],
            "C_ii" : [0.0000005, 0.0000005],
            "W_e" : [50.0, 50.0],
            "W_i" : [0.0, 0.0],
            "X_e" : [1.0, 1.0],
            "Y_e" : [0.0, 0.0],
            "X_i" : [1.0, 1.0],
            "Y_i" : [0.0, 0.0],
            "noise" : [0.0, 0.0],
            "stimulus" : [0.0, 0.0]
        }
    }
}

In [ ]:
# this is from Zerlaut's repo
# so that I can compare my fitting results
# with his ones directly

# NOTE: Zerlaut originally had a mistake in get_fluct_regime_vars function

import numpy as np
import scipy.special as sp_spec
import scipy.integrate as sp_int
from scipy.optimize import minimize, curve_fit
import sys

def pseq_params(params):
    Qe, Te, Ee = params['Qe'], params['Te'], params['Ee']
    Qi, Ti, Ei = params['Qi'], params['Ti'], params['Ei']
    Gl, Cm , El = params['Gl'], params['Cm'] , params['El']
    for key, dval in zip(['Ntot', 'pconnec', 'gei'], [1, 2., 0.5]):
        if not key in params.keys():
            params[key] = dval

    if 'P' in params.keys():
        P = params['P']
    else: # no correction
        P = [-45e-3]
        for i in range(1,11):
            P.append(0)

    # return Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10
    return params['Qe'], params['Te'], params['Ee'], params['Qi'], params['Ti'], params['Ei'], params['Gl'], params['Cm'], params['El'], params['Ntot'], params['pconnec'], params['gei'], P[0], P[1], P[2], P[3], P[4], P[5], P[6], P[7], P[8], P[9], P[10]

def get_fluct_regime_vars(Fe, Fi, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    fe = Fe*(1.-gei)*pconnec*Ntot # default is 1 !!
    fi = Fi*gei*pconnec*Ntot
    
    muGe, muGi = Qe*Te*fe, Qi*Ti*fi
    muG = Gl+muGe+muGi
    muV = (muGe*Ee+muGi*Ei+Gl*El)/muG
    muGn, Tm = muG/Gl, Cm/muG
    
    Ue, Ui = Qe/muG*(Ee-muV), Qi/muG*(Ei-muV)

    sV = np.sqrt(\
                 fe*(Ue*Te)**2/2./(Te+Tm)+\
                 fi*(Ui*Ti)**2/2./(Ti+Tm))
                #  fi*(Qi*Ui)**2/2./(Ti+Tm))

    # NOTE: Zerlaut original code
    # fe, fi = fe+1e-9, fi+1e-9 # just to insure a non zero division, 
    # Tv = ( fe*(Ue*Te)**2 + fi*(Ui*Ti)**2 ) /( fe*(Ue*Te)**2/(Te+Tm) + fi*(Ui*Ti)**2/(Ti+Tm) )
    # # Tv = ( fe*(Ue*Te)**2 + fi*(Qi*Ui)**2 ) /( fe*(Ue*Te)**2/(Te+Tm) + fi*(Qi*Ui)**2/(Ti+Tm) )
    # TvN = Tv*Gl/Cm
    # return muV, sV+1e-12, muGn, TvN

    te = fe * (Ue * Te) ** 2
    ti = fi * (Ui * Ti) ** 2
    te[te<1e-12] = 1e-12
    ti[ti<1e-12] = 1e-12
    Tv = (te + ti) / (te / (Te + Tm) + ti / (Ti + Tm))
    TvN = Tv * Gl / Cm

    sV[sV<1e-12] = 1e-12

    return muV, sV, muGn, TvN




def mean_and_var_conductance(Fe, Fi, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    fe = Fe*(1.-gei)*pconnec*Ntot # default is 1 !!
    fi = Fi*gei*pconnec*Ntot
    return Qe*Te*fe, Qi*Ti*fi, Qe*np.sqrt(Te*fe/2.), Qi*np.sqrt(Ti*fi/2.)


### FUNCTION, INVERSE FUNCTION
def erfc_func(muV, sV, TvN, Vthre, Gl, Cm):
    return .5/TvN*Gl/Cm*\
      sp_spec.erfc((Vthre-muV)/np.sqrt(2)/sV)

def effective_Vthre(Y, muV, sV, TvN, Gl, Cm):
    Vthre_eff = muV+np.sqrt(2)*sV*sp_spec.erfcinv(\
                    Y*2.*TvN*Cm/Gl) # effective threshold
    return Vthre_eff

def threshold_func(muV, sV, TvN, muGn, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    """
    setting by default to True the square
    because when use by external modules, coeff[5:]=np.zeros(3)
    in the case of a linear threshold
    """
    muV0, DmuV0 = -60e-3,10e-3
    sV0, DsV0 =4e-3, 6e-3
    TvN0, DTvN0 = 0.5, 1.
    return P0+P1*(muV-muV0)/DmuV0+\
        P2*(sV-sV0)/DsV0+P3*(TvN-TvN0)/DTvN0+\
        P4*np.log(muGn)+P5*((muV-muV0)/DmuV0)**2+\
        P6*((sV-sV0)/DsV0)**2+P7*((TvN-TvN0)/DTvN0)**2+\
        P8*(muV-muV0)/DmuV0*(sV-sV0)/DsV0+\
        P9*(muV-muV0)/DmuV0*(TvN-TvN0)/DTvN0+\
        P10*(sV-sV0)/DsV0*(TvN-TvN0)/DTvN0
      
# final transfer function template :
def TF_my_template(fe, fi, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    # here TOTAL (sum over synapses) excitatory and inhibitory input
    muV, sV, muGn, TvN = get_fluct_regime_vars(fe, fi, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)
    Vthre = threshold_func(muV, sV, TvN, muGn, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)
    Fout_th = erfc_func(muV, sV, TvN, Vthre, Gl, Cm)
    return Fout_th

def make_loop(t, nu, vm, nu_aff_exc, nu_aff_inh, BIN,\
              Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10):
    dt = t[1]-t[0]
    # constructing the Euler method for the activity rate
    for i_t in range(len(t)-1): # loop over time
        
        fe = (nu_aff_exc[i_t]+nu[i_t]+Fdrive) # afferent+recurrent excitation
        fi = nu[i_t]+nu_aff_inh[i_t] # recurrent inhibition
        W[i_t+1] = W[i_t] + dt/Tw*(b*nu[i_t]*Tw - W[i_t])

        nu[i_t+1] = nu[i_t] +\
               dt/BIN*(\
                TF_my_template(fe, fi, W[i_t], Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)\
                -nu[i_t])

        vm[i_t], _, _, _ = get_fluct_regime_vars(fe, fi, W[i_t], Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, P0, P1, P2, P3, P4, P5, P6, P7, P8, P9, P10)

    return nu, vm, W


################################################################
##### Now fitting to Transfer Function data
################################################################


def fitting_Vthre_then_Fout(Fout, Fe_eff, fiSim, params,\
                            maxiter=10000, xtol=1e-5,
                            verbose=False,
                            with_square_terms=False):

    Fout, Fe_eff, fiSim = Fout.flatten(), Fe_eff.flatten(), fiSim.flatten()
    
    muV, sV, muGn, TvN = get_fluct_regime_vars(Fe_eff, fiSim, *pseq_params(params))
    i_non_zeros = np.where(Fout>0)

    Vthre_eff = effective_Vthre(Fout[i_non_zeros], muV[i_non_zeros],\
                sV[i_non_zeros], TvN[i_non_zeros], params['Gl'], params['Cm'])
    
    if with_square_terms:
        P = np.zeros(11)
    else:
        P = np.zeros(5)
    P[:5] = Vthre_eff.mean(), 1e-3, 1e-3, 1e-3, 1e-3

    def Res(p):
        if not with_square_terms:
            pp = np.concatenate([p, np.zeros(6)])
        else:
            pp=p
        vthre = threshold_func(muV[i_non_zeros], sV[i_non_zeros],\
                               TvN[i_non_zeros], muGn[i_non_zeros], *pp)
        return np.mean((Vthre_eff-vthre)**2)*1e6
    
    plsq = minimize(Res, P, method='SLSQP',\
                    options={'ftol': 1e-8, 'disp': True, 'maxiter':40000})

    if verbose:
        print(plsq)

    P = plsq.x
    print('Fitted Vthre params: ', [x*1e3 for x in P])
    
    def Res(p):
        if not with_square_terms:
            params['P'] = np.concatenate([p, np.zeros(6)])
        else:
            params['P'] = p
        return np.mean((Fout-\
                        TF_my_template(Fe_eff, fiSim, *pseq_params(params)))**2)

    plsq = minimize(Res, P, method='nelder-mead',\
            options={'xtol': xtol, 'disp': True, 'maxiter':maxiter})

    print('Fitted Fout params: ', [x*1e3 for x in plsq.x])

    if verbose:
        print(plsq)
    
    if with_square_terms:
        return plsq.x
    else:
        return np.concatenate([plsq.x, np.zeros(6)])

def make_fit_from_data(DATA, with_square_terms=False,
                       verbose=False):

    MEANfreq, SDfreq, Fe_eff, fiSim, params = np.load(DATA, allow_pickle=True)

    Fe_eff, Fout = np.array(Fe_eff), np.array(MEANfreq)
    levels = fiSim # to store for colors
    fiSim = np.meshgrid(np.zeros(Fe_eff.shape[1]), fiSim)[1]

    P = fitting_Vthre_then_Fout(Fout, Fe_eff, fiSim, params,\
                                with_square_terms=with_square_terms,
                                verbose=verbose)
                            
    print('==================================================')
    print(1e3*np.array(P), 'mV')

    # then we save it:
    # filename = DATA.replace('.npy', '_fit.npy')
    # print('coefficients saved in ', filename)
    # np.save(filename, np.array(P))

    return P

    



In [ ]:
# This cell sets up the data structure for use of MFT

from codes.data_structures.base import Results

# Building a data structure for storing the results
# Simplified version


class SingleNeuronSimpleResults(Results):
    """Data structure for storing results from a single neuron simulation.
    
    That means it contains the stationary results for constant synaptic input

    Units
    -----
    - nu_e, nu_i : [Hz]
    - nu_out : [Hz]
    - w_mean, w_std : [pA]
    - v_mean, v_std : [mV]
    - v_tau : [ms]
    - gsyn_e_mean, gsyn_e_std : [nS]
    - gsyn_i_mean, gsyn_i_std : [nS]

    """

    def __init__(self, 
                 neuron_name, 
                 neuron_params, 
                 nu_out_mean,
                 nu_out_std,
                 nu_e,
                 nu_i,
                 ):
        self.neuron_name = neuron_name
        self.neuron_params = neuron_params

        # 2D grid (nu_e, nu_i)
        self.nu_e = nu_e
        self.nu_i = nu_i
        self.nu_out_mean = nu_out_mean
        self.nu_out_std = nu_out_std

        # Updated naming to match the new structure
        # All the results are 2D arrays with the indexing (exc_rate, inh_rate)
        # Mean synaptic drive used in the trials
        self.exc_drive_mean = nu_e
        self.inh_drive_mean = nu_i

        # In the following the mean and std are computed first over stationary time
        # and then over trials
        self.out_rate_mean = nu_out_mean
        self.out_rate_std = nu_out_std




# Here creating instance of the workflow runner, just to get the parameters

runner = rw.WorkflowRunner(sim_name="DiVolo-Static",
                           snn_params="zerlaut-static_network.yaml",
                           workflow_params=workflow_params,
                           stimuli_params="divolo-static_stimuli.yaml",
                           neuron_data_file_mark="zerlaut-static",
                           results_dir="zerlaut-static_results",
                           )
runner.add_default_mf_model("MF default")

runner.mf_tf_sim_pars[0] # parameters for transfer function fitting
runner.mf_net_pars[0]  # parameters of the network

# here I load zerlaut's data
neuron = 'exc_neuron'

if neuron == 'exc_neuron':
    data_file = 'RS-cell_CONFIG1.npy'
elif neuron == 'inh_neuron':
    data_file = 'FS-cell_CONFIG1.npy'

data_path_repo = Path('../projects/01_fitting_tf/zerlaut2018_data')
data_path_generated = Path('../projects/01_fitting_tf/zerlaut2018_data_generated')

data_path = data_path_repo

zerlaut_data_path = data_path / data_file

rs_data = np.load(zerlaut_data_path, allow_pickle=True)

nu_out_mean = rs_data[0]  # (10, 10) array, Mean output firing rate (Hz)
nu_out_std = rs_data[1]  # (10, 10) array, STD firing rate (Hz)
nu_exc = rs_data[2]  # (10, 10) array, Excitatory input (Hz)
nu_inh = np.stack([rs_data[3]]*10, axis=1)  # (10, 10) array, Inhibitory input (Hz)
params = rs_data[4]  # dict with params (SI units)

# (row, col) == (inh_idx, exc_idx)

# use approximate comparison instead of exact equality
rtol, atol = 1e-6, 1e-12

def assert_close(a, b, name="", rtol=rtol, atol=atol):
    if not np.allclose(a, b, rtol=rtol, atol=atol):
        raise AssertionError(f"{name} differ: {a} vs {b}")

Qe = runner.mf_net_pars[0]['synapses']['exc_synapses']['syn_params']['weight'] *1e-9
Te = runner.mf_net_pars[0][neuron]['neuron_params']['tau_syn_E'] * 1e-3
Ee = runner.mf_net_pars[0][neuron]['neuron_params']['e_rev_E'] * 1e-3

Qi = runner.mf_net_pars[0]['synapses']['inh_synapses']['syn_params']['weight'] * 1e-9
Ti = runner.mf_net_pars[0][neuron]['neuron_params']['tau_syn_I'] * 1e-3
Ei = runner.mf_net_pars[0][neuron]['neuron_params']['e_rev_I'] * 1e-3

Cm = runner.mf_net_pars[0][neuron]['neuron_params']['cm'] * 1e-9
Gl = Cm / (runner.mf_net_pars[0][neuron]['neuron_params']['tau_m'] * 1e-3)
El = runner.mf_net_pars[0][neuron]['neuron_params']['v_rest'] * 1e-3

Ntot = runner.mf_net_pars[0]['network']['total_pop_size']
pconnec = runner.mf_net_pars[0]['network']['p_connect_exc']
gei = runner.mf_net_pars[0]['network']['g']

assert_close((Qe, Te, Ee), (params['Qe'], params['Te'], params['Ee']), "exc syn params")
assert_close((Qi, Ti, Ei), (params['Qi'], params['Ti'], params['Ei']), "inh syn params")
assert_close((Gl, Cm, El), (params['Gl'], params['Cm'], params['El']), "passive params")

neuron_results ={
"exc_neuron": SingleNeuronSimpleResults(
    neuron_name="Zerlaut_RS_CONFIG1",
    neuron_params=runner.mf_net_pars[0][neuron],
    nu_out_mean=nu_out_mean.T,
    nu_out_std=nu_out_std.T,
    nu_e=nu_exc.T,
    nu_i=nu_inh.T,
    )
}

In [ ]:
# just to check that the parameters match 
# there is difference on the level of 1e-12 which is negligible and can be due to different precision in the original data and in the parameters I use here

print(Qe == params['Qe'], Te == params['Te'], Ee == params['Ee'])
print(Qi == params['Qi'], Ti == params['Ti'], Ei == params['Ei'])
print(Gl == params['Gl'], Cm == params['Cm'], El == params['El'])
print(Gl, params['Gl'], Cm , params['Cm'])
print(Gl- params['Gl'])

In [ ]:
zerlaut_fit = make_fit_from_data(zerlaut_data_path, with_square_terms=True)
rs_data_fit = np.load(data_path_repo / "RS-cell_CONFIG1_fit.npy", allow_pickle=True)

print("Script fit", zerlaut_fit*1e3, "", sep="\n")
print("Repo fit", rs_data_fit*1e3, "", sep="\n")

my_tfs = tf.run_fitting_workflow(
    runner.mf_tf_sim_pars[0],
    runner.mf_net_pars[0],
    neuron_results,
    Path("../zerlaut2018_data/my_fit.json")
)

zerlaut_tf = tf.TransferFunction(
                zerlaut_fit*1e3,
                runner.mf_net_pars[0]["transfer_function"]["expansion_point"],
                runner.mf_net_pars[0]["transfer_function"]["expansion_norm"],
                runner.mf_net_pars[0][neuron],
                square_terms=runner.mf_tf_sim_pars[0]["square_terms"], 
                log_term=runner.mf_tf_sim_pars[0]["log_term"])

repo_tf = tf.TransferFunction(
                rs_data_fit*1e3,
                runner.mf_net_pars[0]["transfer_function"]["expansion_point"],
                runner.mf_net_pars[0]["transfer_function"]["expansion_norm"],
                runner.mf_net_pars[0][neuron],
                square_terms=runner.mf_tf_sim_pars[0]["square_terms"], 
                log_term=runner.mf_tf_sim_pars[0]["log_term"])


my_tf = my_tfs[neuron]

print("Zerlaut's fit coefficients (mV): ", zerlaut_fit*1e3)
print("My fit coefficients (mV): ", my_tfs[neuron].v_eff.coefs)


In [ ]:
zerlaut_v_eff_fit = [-47.52402120189918, 4.3586568814149675, -5.896946682172716, -4.2856454833992075, -1.4403870587883356, 0.7818045165366436, 1.3222448324419873, -9.881533729669776, 2.876420486035771, 0.9754971140220596, -1.0683774021766608]
# zerlaut_v_eff_fit = [-48.28573709, 3.96820883,-3.24738656,-2.07683977,-0.71426687, 0.88598422, 2.62370942, -10.46193566, 2.44841237, 1.4020124,  -2.25671209]
tf_test = tf.TransferFunction(
    zerlaut_v_eff_fit, 
    runner.mf_net_pars[0]['transfer_function']['expansion_point'], 
    runner.mf_net_pars[0]['transfer_function']['expansion_norm'], 
    runner.mf_net_pars[0][neuron], 
    square_terms=True, 
    log_term=True)

tf_test.make_fit(nu_out_mean.T, nu_exc.T, nu_inh.T, method='nelder-mead', options={'xatol': 1e-5, 'disp': True, 'maxiter':10000}, flattened=False)

tf_test.v_eff.coefs


In [ ]:
# runner.mf_tf_sim_pars[0] # parameters for transfer function fitting
runner.mf_net_pars[0]['transfer_function']  # parameters of the network


In [ ]:
# this calls the fitting workflow

step=1

for inh_idx, inh_input in enumerate(nu_inh[:,0]):
    if inh_idx % step != 0:
        continue

    # Plotting the data points
    plt.errorbar(nu_exc[inh_idx], nu_out_mean[inh_idx], yerr=nu_out_std[inh_idx], fmt="o", label=f"nu_inh={inh_input:.1f} Hz")

    # My TF fit
    plt.plot(nu_exc[inh_idx], my_tf(nu_exc[inh_idx], inh_input), ":", color='black')

    # Zerlaut TF fit with my implementation
    plt.plot(nu_exc[inh_idx], zerlaut_tf(nu_exc[inh_idx], inh_input), "--", color='gray')

    # Zerlaut TF fit with Zerlaut's implementation
    plt.plot(nu_exc[inh_idx], TF_my_template(
                                nu_exc[inh_idx], 
                                inh_input, 
                                Qe, 
                                Te, 
                                Ee, 
                                Qi, 
                                Ti, 
                                Ei, 
                                Gl, 
                                Cm, 
                                El, 
                                Ntot, 
                                pconnec, 
                                gei, 
                                *zerlaut_fit),
                "--", color='blue'
    )

    # Zerlaut repo fit
    # plt.plot(nu_exc[inh_idx], repo_tf(nu_exc[inh_idx], inh_input), "-.", color='green')
    plt.plot(nu_exc[inh_idx], TF_my_template(
                                nu_exc[inh_idx], 
                                inh_input, 
                                Qe, 
                                Te, 
                                Ee, 
                                Qi, 
                                Ti, 
                                Ei, 
                                Gl, 
                                Cm, 
                                El, 
                                Ntot, 
                                pconnec, 
                                gei, 
                                *rs_data_fit),
                "--", color='green'
    )
plt.legend()

Now it looks like, that my and Zerlaut's script give the same output, but the fitted coefficients still does not match!

Moreover, the provided fit params in the repo are total not corresponding!



In [ ]:
fig, ax = plt.subplots(nrows=5, ncols=1, figsize=(8,16))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for inh_idx, inh_input in enumerate(nu_inh[:,0]):
    if inh_idx % step != 0:
        continue

    # cycle through default matplotlib colors
    color = colors[inh_idx % len(colors)]

    # zerlauts results
    muV, sV, muGn, TvN = get_fluct_regime_vars(nu_exc[inh_idx], inh_input, Qe, Te, Ee, Qi, Ti, Ei, Gl, Cm, El, Ntot, pconnec, gei, *zerlaut_fit)

    i_non_zeros = np.where(nu_exc[inh_idx]>0)
    Vthre_eff = effective_Vthre(nu_exc[inh_idx][i_non_zeros], muV[i_non_zeros], sV[i_non_zeros], TvN[i_non_zeros], params['Gl'], params['Cm'])


    # my results
    mu_V, sigma_V, tau_V, tau_VN, mu_G = tf.MeanPotentialFluctuations(runner.mf_net_pars[0][neuron])(nu_exc[inh_idx], inh_input)

    v_eff = tf.V_eff.v_eff_from_data(nu_exc[inh_idx][i_non_zeros], mu_V[i_non_zeros], sigma_V[i_non_zeros], tau_V[i_non_zeros], tau_VN[i_non_zeros])

    ax[0].plot(nu_exc[inh_idx], muV*1e3, 'x', color=color, label=fr"$\nu_{{inh}}={inh_input:.1f} Hz$")
    ax[0].plot(nu_exc[inh_idx], mu_V, 'o', color=color)
    print(f"Total difference in muV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muV*1e3 - mu_V)), "mV")

    ax[1].plot(nu_exc[inh_idx], sV*1e3, 'x', color=color)
    ax[1].plot(nu_exc[inh_idx], sigma_V, 'o', color=color)
    print(f"Total difference in sigmaV for inh={inh_input:.1f} Hz : ", np.mean(np.abs(sV*1e3 - sigma_V)), "mV")

    ax[2].plot(nu_exc[inh_idx], TvN, 'x', color=color)
    ax[2].plot(nu_exc[inh_idx], tau_VN, 'o', color=color)
    print(f"Total difference in tauVN for inh={inh_input:.1f} Hz : ", np.mean(np.abs(TvN - tau_VN)), "ms")
    
    ax[3].plot(nu_exc[inh_idx], muGn*1e9, 'x', color=color)
    ax[3].plot(nu_exc[inh_idx], mu_G/Gl, 'o', color=color)
    print(f"Total difference in muG for inh={inh_input:.1f} Hz : ", np.mean(np.abs(muGn*1e9 - mu_G/Gl)), "nS")

    ax[4].plot(nu_exc[inh_idx][i_non_zeros], Vthre_eff*1e3, 'x', color=color)
    ax[4].plot(nu_exc[inh_idx][i_non_zeros], v_eff, 'o', color=color)
    print(f"Total difference in Vthre_eff for inh={inh_input:.1f} Hz : ", np.mean(np.abs(Vthre_eff*1e3 - v_eff)), "mV")

ax[0].set_ylabel(r"$\mu_V$ (mV)")
ax[1].set_ylabel(r"$\sigma_V$ (mV)")
ax[2].set_ylabel(r"$\tau_V^N$ (ms)")
ax[3].set_ylabel(r"$\mu_{G}/G_l$ (nS)")
ax[4].set_ylabel(r"$V_{thre}^{eff}$ (mV)")
ax[-1].set_xlabel(r"$\nu_{exc}$ (Hz)")

# plt.legend()

In [ ]:
data_path_repo = Path('../projects/01_fitting_tf/zerlaut2018_data')
data_path_generated = Path('../projects/01_fitting_tf/zerlaut2018_data_generated')

rs_data_fit = np.load(data_path_repo / "RS-cell_CONFIG1_fit.npy", allow_pickle=True)
# rs_data_fit = np.load(data_path_generated / "RS-cell_CONFIG1_fit.npy", allow_pickle=True)
rs_data_fit*1e3